# Day 3: Pandas Data Wrangling

**Python for Data Science Workshop**  
Dr. Yaé Ulrich Gaba

---

## Learning Objectives

By the end of this session you will be able to:

1. Create and inspect Series and DataFrames
2. Select, filter, and sort data with `loc`, `iloc`, boolean indexing, and `query()`
3. Engineer new columns using `apply()`, `map()`, and `.str` methods
4. Identify and handle missing values and duplicates
5. Aggregate data with `groupby`, `agg`, `transform`, and `pivot_table`
6. Combine datasets with `merge`, `join`, and `concat`

---

In [ ]:
import pandas as pd
import numpy as np
import io

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"pandas {pd.__version__}")
print(f"numpy  {np.__version__}")

---
## 1. Pandas Basics

Pandas provides two core data structures:

| Structure | Dimensions | Analogy |
|-----------|-----------|----------|
| `Series`  | 1-D | A single column / array with labels |
| `DataFrame` | 2-D | A table / spreadsheet |

### 1.1 Series

In [ ]:
# Creating a Series from a list
populations = pd.Series(
    [213.4, 206.1, 114.9, 58.6, 54.0],
    index=["Nigeria", "Ethiopia", "Egypt", "Tanzania", "South Africa"],
    name="population_millions"
)
print(populations)
print(f"\nDtype : {populations.dtype}")
print(f"Shape : {populations.shape}")
print(f"Index : {populations.index.tolist()}")

In [ ]:
# Creating a Series from a dictionary
gdp_per_capita = pd.Series({
    "Nigeria": 2066,
    "Ethiopia": 925,
    "Egypt": 3699,
    "Tanzania": 1099,
    "South Africa": 6994
}, name="gdp_per_capita_usd")

gdp_per_capita

In [ ]:
# Vectorised arithmetic on Series
total_gdp_billions = (populations * gdp_per_capita) / 1000
total_gdp_billions.name = "total_gdp_billions_usd"
print(total_gdp_billions)

### 1.2 DataFrame — from dictionaries and lists

In [ ]:
# From a dictionary of lists
df_dict = pd.DataFrame({
    "country": ["Nigeria", "Ethiopia", "Egypt", "Tanzania", "South Africa"],
    "population_m": [213.4, 206.1, 114.9, 58.6, 54.0],
    "gdp_per_capita": [2066, 925, 3699, 1099, 6994],
    "life_expectancy": [52.7, 65.0, 71.8, 65.5, 64.1]
})
df_dict

In [ ]:
# From a list of dictionaries (row-oriented)
rows = [
    {"city": "Lagos", "country": "Nigeria", "pop_m": 15.4},
    {"city": "Cairo", "country": "Egypt", "pop_m": 21.3},
    {"city": "Kinshasa", "country": "DRC", "pop_m": 14.3},
    {"city": "Johannesburg", "country": "South Africa", "pop_m": 5.8},
]
df_rows = pd.DataFrame(rows)
df_rows

### 1.3 Reading CSV data (using `io.StringIO` for inline data)

In [ ]:
csv_text = """country,year,gdp_per_capita,population_m,life_expectancy
Nigeria,2019,2230,200.9,54.3
Nigeria,2020,2066,206.1,52.7
Nigeria,2021,2085,211.4,52.4
Kenya,2019,1817,52.6,66.3
Kenya,2020,1838,53.8,66.7
Kenya,2021,2007,54.9,61.4
Ghana,2019,2202,30.4,63.8
Ghana,2020,2328,31.1,63.5
Ghana,2021,2363,31.7,63.1
Egypt,2019,3020,100.4,71.5
Egypt,2020,3699,102.3,71.8
Egypt,2021,3876,104.3,70.2
South Africa,2019,6001,58.8,64.1
South Africa,2020,5091,59.3,62.4
South Africa,2021,6994,60.0,64.1
"""

df = pd.read_csv(io.StringIO(csv_text))
df

### 1.4 Inspecting a DataFrame

In [ ]:
print("--- head(3) ---")
display(df.head(3))

print("\n--- tail(3) ---")
display(df.tail(3))

In [ ]:
print("--- info() ---")
df.info()

In [ ]:
print("--- describe() ---")
display(df.describe())

print("\n--- dtypes ---")
print(df.dtypes)

print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

---
## 2. Selection & Filtering

### 2.1 Selecting columns

In [ ]:
# Single column -> Series
print(df["country"].head())
print()

# Multiple columns -> DataFrame
display(df[["country", "year", "life_expectancy"]].head())

### 2.2 `loc` (label-based) and `iloc` (integer-based)

In [ ]:
# loc: rows by label (here default integer index), columns by name
display(df.loc[0:4, ["country", "gdp_per_capita"]])

print()

# iloc: purely positional
display(df.iloc[0:3, 0:3])  # first 3 rows, first 3 columns

### 2.3 Boolean indexing

In [ ]:
# Rows where life expectancy > 65
high_le = df[df["life_expectancy"] > 65]
display(high_le)

print()

# Combining conditions (use & for AND, | for OR, ~ for NOT)
mask = (df["year"] == 2021) & (df["gdp_per_capita"] > 2500)
display(df[mask])

### 2.4 `query()` — readable string-based filtering

In [ ]:
# Same filter using query()
df.query("year == 2021 and gdp_per_capita > 2500")

In [ ]:
# Using variables inside query with @
min_pop = 60
df.query("population_m >= @min_pop")

### 2.5 Sorting

In [ ]:
# Sort by GDP per capita descending
df.sort_values("gdp_per_capita", ascending=False).head(5)

In [ ]:
# Multi-column sort
df.sort_values(["country", "year"], ascending=[True, False]).head(6)

---
## 3. Column Operations

### 3.1 Creating new columns

In [ ]:
# Total GDP = GDP per capita * population (in billions)
df["total_gdp_bn"] = (df["gdp_per_capita"] * df["population_m"]) / 1000

# Log of GDP per capita
df["log_gdp"] = np.log(df["gdp_per_capita"])

df.head()

### 3.2 `apply()` — row-wise or column-wise custom functions

In [ ]:
def income_category(gdp):
    """World Bank-style income classification (simplified)."""
    if gdp < 1086:
        return "Low income"
    elif gdp < 4255:
        return "Lower-middle income"
    elif gdp < 13205:
        return "Upper-middle income"
    else:
        return "High income"

df["income_group"] = df["gdp_per_capita"].apply(income_category)
df[["country", "year", "gdp_per_capita", "income_group"]].head(6)

In [ ]:
# apply() with a lambda across an entire row (axis=1)
df["summary"] = df.apply(
    lambda row: f"{row['country']} ({row['year']}): ${row['gdp_per_capita']:,}/cap",
    axis=1
)
df["summary"].head(6)

### 3.3 `map()` — element-wise mapping

In [ ]:
region_map = {
    "Nigeria": "West Africa",
    "Ghana": "West Africa",
    "Kenya": "East Africa",
    "Egypt": "North Africa",
    "South Africa": "Southern Africa"
}

df["region"] = df["country"].map(region_map)
df[["country", "region"]].drop_duplicates()

### 3.4 String methods with `.str` accessor

In [ ]:
print("Upper-case country names:")
print(df["country"].str.upper().unique())

print("\nCountries containing 'a' (case-insensitive):")
print(df.loc[df["country"].str.contains("a", case=False), "country"].unique())

print("\nFirst 3 characters:")
print(df["country"].str[:3].unique())

print("\nCharacter count:")
print(df["country"].str.len().unique())

---
## 4. Data Cleaning

### 4.1 Build a messy dataset to practise on

In [ ]:
messy_csv = """country,year,gdp_per_capita,population_m,life_expectancy
Nigeria,2020,2066,206.1,52.7
Nigeria,2020,2066,206.1,52.7
Kenya,2020,,53.8,66.7
Ghana,2020,2328,31.1,
Egypt,2020,3699,,71.8
south africa,2020,5091,59.3,62.4
Tanzania,2020,1076,59.7,65.5
Tanzania,2020,1076,59.7,65.5
Rwanda,2020,797,13.0,
Senegal,2020,1487,,66.8
"""

dirty = pd.read_csv(io.StringIO(messy_csv))
dirty

### 4.2 Detecting missing values

In [ ]:
print("Missing values per column:")
print(dirty.isna().sum())

print(f"\nTotal missing cells: {dirty.isna().sum().sum()}")
print(f"Rows with any NaN:  {dirty.isna().any(axis=1).sum()}")

# Percentage missing
print("\nPercentage missing:")
print((dirty.isna().mean() * 100).round(1))

### 4.3 Handling missing values — `fillna` and `dropna`

In [ ]:
# Fill GDP with median
filled = dirty.copy()
filled["gdp_per_capita"] = filled["gdp_per_capita"].fillna(
    filled["gdp_per_capita"].median()
)

# Fill life expectancy with the column mean
filled["life_expectancy"] = filled["life_expectancy"].fillna(
    filled["life_expectancy"].mean().round(1)
)

print("After filling GDP and life expectancy:")
print(filled.isna().sum())
display(filled)

In [ ]:
# dropna: drop rows that still have any NaN
cleaned = filled.dropna()
print(f"Rows before dropna: {len(filled)}, after: {len(cleaned)}")
display(cleaned)

### 4.4 Duplicates

In [ ]:
print("Duplicated rows:")
display(cleaned[cleaned.duplicated(keep=False)])

cleaned = cleaned.drop_duplicates()
print(f"\nAfter dedup: {len(cleaned)} rows")
display(cleaned)

### 4.5 Type conversion with `astype` and standardising text

In [ ]:
# Standardise country names to title case
cleaned["country"] = cleaned["country"].str.title()

# Ensure year is integer
cleaned["year"] = cleaned["year"].astype(int)

print(cleaned.dtypes)
print()
display(cleaned)

---
## 5. Aggregation

We will switch back to the main multi-year dataset for richer aggregation examples.

In [ ]:
# Reload the clean multi-year data
df = pd.read_csv(io.StringIO(csv_text))
df["total_gdp_bn"] = (df["gdp_per_capita"] * df["population_m"]) / 1000
df.head()

### 5.1 `groupby` basics

In [ ]:
# Mean GDP per capita by country
df.groupby("country")["gdp_per_capita"].mean().round(0)

In [ ]:
# Multiple aggregations at once
df.groupby("country")["life_expectancy"].agg(["mean", "min", "max"]).round(1)

### 5.2 `agg` with multiple functions per column

In [ ]:
summary = df.groupby("country").agg(
    avg_gdp=("gdp_per_capita", "mean"),
    max_gdp=("gdp_per_capita", "max"),
    avg_life_exp=("life_expectancy", "mean"),
    total_pop_latest=("population_m", "last"),
).round(1)

summary.sort_values("avg_gdp", ascending=False)

### 5.3 `transform` — broadcast group result back to original rows

In [ ]:
# Each country's GDP deviation from its own mean
df["gdp_country_mean"] = df.groupby("country")["gdp_per_capita"].transform("mean")
df["gdp_deviation"] = df["gdp_per_capita"] - df["gdp_country_mean"]

df[["country", "year", "gdp_per_capita", "gdp_country_mean", "gdp_deviation"]].head(9)

### 5.4 `pivot_table`

In [ ]:
pivot = df.pivot_table(
    index="country",
    columns="year",
    values="gdp_per_capita",
    aggfunc="mean"
)
pivot

In [ ]:
# Pivot table with margins (totals)
df.pivot_table(
    index="country",
    columns="year",
    values="total_gdp_bn",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
).round(1)

---
## 6. Merging Data

### 6.1 Create sample datasets to merge

In [ ]:
# Dataset A: Country metadata
countries = pd.DataFrame({
    "country": ["Nigeria", "Kenya", "Ghana", "Egypt", "South Africa", "Ethiopia"],
    "capital": ["Abuja", "Nairobi", "Accra", "Cairo", "Pretoria", "Addis Ababa"],
    "region": ["West", "East", "West", "North", "Southern", "East"]
})

# Dataset B: Recent economic indicators (2021)
indicators = pd.DataFrame({
    "country": ["Nigeria", "Kenya", "Ghana", "Egypt", "South Africa", "Tanzania"],
    "inflation_pct": [17.0, 6.1, 10.0, 5.2, 4.6, 3.7],
    "unemployment_pct": [33.3, 5.7, 13.4, 7.4, 34.9, 2.7]
})

print("countries:")
display(countries)
print("\nindicators:")
display(indicators)

### 6.2 `merge` — SQL-style joins

In [ ]:
# Inner join (only countries in BOTH datasets)
inner = pd.merge(countries, indicators, on="country", how="inner")
print("INNER JOIN:")
display(inner)

In [ ]:
# Left join (keep all countries from left table)
left = pd.merge(countries, indicators, on="country", how="left")
print("LEFT JOIN:")
display(left)

In [ ]:
# Outer join (keep everything)
outer = pd.merge(countries, indicators, on="country", how="outer")
print("OUTER JOIN:")
display(outer)

### 6.3 `concat` — stacking DataFrames

In [ ]:
# Vertical concatenation (stacking rows)
batch_1 = pd.DataFrame({"country": ["Nigeria", "Kenya"], "score": [72, 68]})
batch_2 = pd.DataFrame({"country": ["Ghana", "Egypt"], "score": [65, 78]})

combined = pd.concat([batch_1, batch_2], ignore_index=True)
print("Vertical concat:")
display(combined)

In [ ]:
# Horizontal concatenation (side by side)
names = pd.DataFrame({"country": ["Nigeria", "Kenya", "Ghana"]})
scores = pd.DataFrame({"hdi": [0.539, 0.601, 0.611]})

side_by_side = pd.concat([names, scores], axis=1)
print("Horizontal concat:")
display(side_by_side)

### 6.4 `join` — index-based merge

In [ ]:
# Set country as index for both frames, then join
c_indexed = countries.set_index("country")
i_indexed = indicators.set_index("country")

joined = c_indexed.join(i_indexed, how="inner")
print("Index-based join:")
display(joined)

---
## 7. Lab 3 — African Development Data Analysis

In this lab you will work with **realistic African development data** spanning 10 countries and 5 years. The data arrives in three separate "files" (inline DataFrames) that you must clean, merge, and analyse.

### Scenario

You are a data analyst at the **African Development Bank**. Three different departments have sent you data:

1. **GDP data** — GDP per capita (current USD) for 10 countries, 2018–2022
2. **Demographics data** — population and life expectancy (with some missing values and duplicates)
3. **Country metadata** — region and income classification

Your task: produce a clean, merged dataset and generate summary statistics.

### Step 1: Create the raw datasets

In [ ]:
# ---- Dataset 1: GDP per capita (USD) ----
gdp_data = pd.DataFrame({
    "country": [
        "Nigeria", "Nigeria", "Nigeria", "Nigeria", "Nigeria",
        "Kenya", "Kenya", "Kenya", "Kenya", "Kenya",
        "Ghana", "Ghana", "Ghana", "Ghana", "Ghana",
        "Egypt", "Egypt", "Egypt", "Egypt", "Egypt",
        "South Africa", "South Africa", "South Africa", "South Africa", "South Africa",
        "Ethiopia", "Ethiopia", "Ethiopia", "Ethiopia", "Ethiopia",
        "Tanzania", "Tanzania", "Tanzania", "Tanzania", "Tanzania",
        "Rwanda", "Rwanda", "Rwanda", "Rwanda", "Rwanda",
        "Senegal", "Senegal", "Senegal", "Senegal", "Senegal",
        "Cote d'Ivoire", "Cote d'Ivoire", "Cote d'Ivoire", "Cote d'Ivoire", "Cote d'Ivoire"
    ],
    "year": [2018, 2019, 2020, 2021, 2022] * 10,
    "gdp_per_capita": [
        2028, 2230, 2066, 2085, 2184,   # Nigeria
        1711, 1817, 1838, 2007, 2099,   # Kenya
        2202, 2202, 2328, 2363, 2363,   # Ghana
        2573, 3020, 3699, 3876, 3698,   # Egypt
        6374, 6001, 5091, 6994, 6469,   # South Africa
        772,  856,  936,  925,  1028,   # Ethiopia
        1044, 1076, 1076, 1099, 1192,   # Tanzania
        773,  797,  780,  834,  966,    # Rwanda
        1446, 1487, 1488, 1637, 1669,   # Senegal
        2286, 2286, 2280, 2486, 2549    # Cote d'Ivoire
    ]
})

print(f"GDP data shape: {gdp_data.shape}")
gdp_data.head()

In [ ]:
# ---- Dataset 2: Demographics (intentionally messy) ----
demo_csv = """country,year,population_m,life_expectancy
Nigeria,2018,195.9,54.0
Nigeria,2019,200.9,54.3
Nigeria,2020,206.1,52.7
Nigeria,2021,211.4,52.4
Nigeria,2022,218.5,53.0
Kenya,2018,51.4,66.1
Kenya,2019,52.6,66.3
Kenya,2020,53.8,66.7
Kenya,2021,54.9,61.4
Kenya,2022,55.0,
Ghana,2018,29.8,63.5
Ghana,2019,30.4,63.8
Ghana,2020,31.1,63.5
Ghana,2021,31.7,63.1
Ghana,2022,32.8,
egypt,2018,98.4,71.2
egypt,2019,100.4,71.5
egypt,2020,102.3,71.8
Egypt,2021,104.3,70.2
Egypt,2022,106.7,70.4
South Africa,2018,57.8,63.9
South Africa,2019,58.8,64.1
South Africa,2020,59.3,62.4
South Africa,2021,60.0,64.1
South Africa,2022,60.6,64.0
Ethiopia,2018,109.2,65.5
Ethiopia,2019,112.1,66.2
Ethiopia,2020,115.0,65.0
Ethiopia,2021,,
Ethiopia,2022,120.3,
Tanzania,2018,56.3,65.0
Tanzania,2019,58.0,65.5
Tanzania,2020,59.7,65.5
Tanzania,2021,61.5,66.2
Tanzania,2022,63.6,66.8
Rwanda,2018,12.3,68.7
Rwanda,2019,12.6,69.0
Rwanda,2020,13.0,68.0
Rwanda,2021,13.3,66.1
Rwanda,2022,13.8,67.5
Senegal,2018,16.0,67.5
Senegal,2019,16.3,67.7
Senegal,2020,16.7,66.8
Senegal,2021,17.2,66.4
Senegal,2022,17.7,66.9
Cote d'Ivoire,2018,25.1,57.4
Cote d'Ivoire,2019,25.7,57.8
Cote d'Ivoire,2020,26.4,57.2
Cote d'Ivoire,2021,27.1,57.5
Cote d'Ivoire,2022,28.0,58.0
Tanzania,2020,59.7,65.5
Rwanda,2019,12.6,69.0
"""

demographics = pd.read_csv(io.StringIO(demo_csv))
print(f"Demographics shape: {demographics.shape}")
print(f"\nMissing values:\n{demographics.isna().sum()}")
print(f"\nDuplicates: {demographics.duplicated().sum()}")

In [ ]:
# ---- Dataset 3: Country metadata ----
metadata = pd.DataFrame({
    "country": [
        "Nigeria", "Kenya", "Ghana", "Egypt", "South Africa",
        "Ethiopia", "Tanzania", "Rwanda", "Senegal", "Cote d'Ivoire"
    ],
    "region": [
        "West Africa", "East Africa", "West Africa", "North Africa", "Southern Africa",
        "East Africa", "East Africa", "East Africa", "West Africa", "West Africa"
    ],
    "income_class": [
        "Lower-middle", "Lower-middle", "Lower-middle", "Lower-middle", "Upper-middle",
        "Low", "Lower-middle", "Low", "Lower-middle", "Lower-middle"
    ]
})
metadata

### Step 2: Clean the demographics data

In [ ]:
# 2a. Standardise country names (fix "egypt" -> "Egypt")
demographics["country"] = demographics["country"].str.title()

# Verify
print("Unique countries after title-case:")
print(demographics["country"].unique())

In [ ]:
# 2b. Remove duplicate rows
print(f"Rows before dedup: {len(demographics)}")
demographics = demographics.drop_duplicates()
print(f"Rows after dedup:  {len(demographics)}")

In [ ]:
# 2c. Handle missing values
print("Missing values before:")
print(demographics.isna().sum())

# Strategy: fill population with forward-fill within each country,
# fill life expectancy with country-level mean
demographics["population_m"] = demographics.groupby("country")["population_m"].transform(
    lambda x: x.ffill()
)

demographics["life_expectancy"] = demographics.groupby("country")["life_expectancy"].transform(
    lambda x: x.fillna(x.mean())
)

print("\nMissing values after:")
print(demographics.isna().sum())

In [ ]:
# 2d. Fix the Cote D'Ivoire title-case issue introduced by str.title()
demographics["country"] = demographics["country"].replace(
    "Cote D'Ivoire", "Cote d'Ivoire"
)

print("Final unique countries:")
print(sorted(demographics["country"].unique()))

### Step 3: Merge all three datasets

In [ ]:
# Merge GDP + Demographics on country and year
merged = pd.merge(gdp_data, demographics, on=["country", "year"], how="inner")

# Merge in metadata on country
merged = pd.merge(merged, metadata, on="country", how="left")

# Compute total GDP
merged["total_gdp_bn"] = (merged["gdp_per_capita"] * merged["population_m"] / 1000).round(2)

print(f"Final merged shape: {merged.shape}")
print(f"Columns: {merged.columns.tolist()}")
display(merged.head(10))

In [ ]:
# Quick data quality check
print("Missing values in merged dataset:")
print(merged.isna().sum())
print(f"\nDuplicates: {merged.duplicated().sum()}")
print(f"Countries:  {merged['country'].nunique()}")
print(f"Years:      {sorted(merged['year'].unique())}")

### Step 4: Summary statistics by country

In [ ]:
country_summary = merged.groupby("country").agg(
    region=("region", "first"),
    income_class=("income_class", "first"),
    avg_gdp_per_capita=("gdp_per_capita", "mean"),
    gdp_growth_rate=("gdp_per_capita", lambda x: ((x.iloc[-1] / x.iloc[0]) - 1) * 100),
    avg_population_m=("population_m", "mean"),
    avg_life_expectancy=("life_expectancy", "mean"),
    avg_total_gdp_bn=("total_gdp_bn", "mean")
).round(1)

country_summary = country_summary.sort_values("avg_gdp_per_capita", ascending=False)
print("=== Country Summary (2018-2022) ===")
display(country_summary)

### Step 5: Summary statistics by year

In [ ]:
year_summary = merged.groupby("year").agg(
    total_population_m=("population_m", "sum"),
    mean_gdp_per_capita=("gdp_per_capita", "mean"),
    median_gdp_per_capita=("gdp_per_capita", "median"),
    total_gdp_bn=("total_gdp_bn", "sum"),
    mean_life_expectancy=("life_expectancy", "mean")
).round(1)

print("=== Yearly Summary (all 10 countries) ===")
display(year_summary)

### Step 6: Summary by region

In [ ]:
region_summary = merged.groupby("region").agg(
    num_countries=("country", "nunique"),
    avg_gdp_per_capita=("gdp_per_capita", "mean"),
    avg_life_expectancy=("life_expectancy", "mean"),
    total_gdp_bn=("total_gdp_bn", "sum")
).round(1)

print("=== Regional Summary ===")
display(region_summary)

### Step 7: Pivot table — GDP per capita by country and year

In [ ]:
gdp_pivot = merged.pivot_table(
    index="country",
    columns="year",
    values="gdp_per_capita",
    aggfunc="mean"
)

# Add a column for 5-year change
if 2022 in gdp_pivot.columns and 2018 in gdp_pivot.columns:
    gdp_pivot["change_pct"] = (
        ((gdp_pivot[2022] - gdp_pivot[2018]) / gdp_pivot[2018]) * 100
    ).round(1)

print("=== GDP per Capita Pivot Table ===")
display(gdp_pivot.sort_values("change_pct", ascending=False))

### Step 8: Key insights

In [ ]:
latest = merged.query("year == 2022")

print("KEY INSIGHTS (2022)")
print("=" * 50)

richest = latest.loc[latest["gdp_per_capita"].idxmax()]
print(f"\nHighest GDP/capita: {richest['country']} (${richest['gdp_per_capita']:,})")

poorest = latest.loc[latest["gdp_per_capita"].idxmin()]
print(f"Lowest GDP/capita:  {poorest['country']} (${poorest['gdp_per_capita']:,})")

longest_life = latest.loc[latest["life_expectancy"].idxmax()]
print(f"\nHighest life expectancy: {longest_life['country']} ({longest_life['life_expectancy']:.1f} years)")

shortest_life = latest.loc[latest["life_expectancy"].idxmin()]
print(f"Lowest life expectancy:  {shortest_life['country']} ({shortest_life['life_expectancy']:.1f} years)")

biggest = latest.loc[latest["total_gdp_bn"].idxmax()]
print(f"\nLargest economy: {biggest['country']} (${biggest['total_gdp_bn']:.1f}B)")

most_pop = latest.loc[latest["population_m"].idxmax()]
print(f"Most populous:   {most_pop['country']} ({most_pop['population_m']:.1f}M)")

total = latest["total_gdp_bn"].sum()
print(f"\nCombined GDP of all 10 countries: ${total:.1f}B")

---

## Summary

| Topic | Key Functions |
|-------|---------------|
| **Create** | `pd.Series()`, `pd.DataFrame()`, `pd.read_csv()` |
| **Inspect** | `.head()`, `.tail()`, `.info()`, `.describe()`, `.dtypes`, `.shape` |
| **Select** | `[]`, `.loc[]`, `.iloc[]`, boolean indexing, `.query()` |
| **Transform** | `.apply()`, `.map()`, `.str` accessor |
| **Clean** | `.isna()`, `.fillna()`, `.dropna()`, `.duplicated()`, `.drop_duplicates()`, `.astype()` |
| **Aggregate** | `.groupby()`, `.agg()`, `.transform()`, `.pivot_table()` |
| **Merge** | `pd.merge()`, `.join()`, `pd.concat()` |

---

**Next session (Day 4):** Data Visualisation with Matplotlib and Seaborn